# Worldle Final Project

This notebook builds a playable Worldle-style game using `ipyleaflet`, `ipywidgets`, and the `wdo` package.

## What I added or finished in `wdo`

- `wdo.geometry.bbox.bbox_from_feature`
- `wdo.geometry.bbox.bbox_from_features`
- `wdo.maps.leaflet_helpers.make_map`
- `wdo.maps.leaflet_helpers.add_geojson`
- `wdo.maps.leaflet_helpers.fit_map_to_geojson`
- `wdo.games.worldle.choose_target`
- `wdo.games.worldle.feature_center`
- `wdo.games.worldle.guess_feedback`
- `wdo.games.worldle.format_feedback`
- `wdo.games.worldle.build_country_lookup`
- `wdo.games.worldle.render_guess_row`
- `wdo.games.worldle.WorldleGame`



In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
RESOURCES = PROJECT_ROOT / "Resources"


if not (RESOURCES / "wdo").exists():
    PROJECT_ROOT = Path.cwd().parent
    RESOURCES = PROJECT_ROOT / "Resources"

if not (RESOURCES / "wdo").exists():
    raise FileNotFoundError("Could not find Resources/wdo. Make sure the notebook is inside the 04-Worldle folder.")

sys.path.insert(0, str(RESOURCES))

import wdo
print("wdo loaded from:", wdo.__file__)

wdo loaded from: /workspaces/ricardoayala2510-Spatial-Data-Mapping/completed_assignments/04-Worldle/Resources/wdo/__init__.py


In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from IPython.display import display
import ipywidgets as widgets

from wdo.io.geojson_tools import load_geojson, get_features, feature_count, property_names
from wdo.maps.leaflet_helpers import make_map, add_geojson, fit_map_to_geojson
from wdo.games.worldle import (
    WorldleGame,
    build_country_lookup,
    format_feedback,
    get_country_name,
    get_iso3,
    render_guess_row,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# Project paths
ROOT = Path.cwd()
DATA_DIR = ROOT / "Resources" / "Data"
FLAG_DIR = DATA_DIR / "flag-icons"

COUNTRY_FILE = DATA_DIR / "countries_export.json"
if not COUNTRY_FILE.exists():
    COUNTRY_FILE = DATA_DIR / "countries.geojson"

FLAG_INDEX_FILE = FLAG_DIR / "country.json"

print("Country file:", COUNTRY_FILE)
print("Flag index:", FLAG_INDEX_FILE)


Country file: /workspaces/ricardoayala2510-Spatial-Data-Mapping/completed_assignments/04-Worldle/Resources/Data/countries.geojson
Flag index: /workspaces/ricardoayala2510-Spatial-Data-Mapping/completed_assignments/04-Worldle/Resources/Data/flag-icons/country.json


In [11]:
# Load country polygons and flag metadata
countries = load_geojson(COUNTRY_FILE)
features_raw = get_features(countries)

with open(FLAG_INDEX_FILE, "r", encoding="utf-8") as f:
    flag_index = json.load(f)

country_lookup, misses = build_country_lookup(
    countries,
    flag_index,
    flag_root=FLAG_DIR,
)

# Keep countries that have usable ISO-3 codes and are in the lookup.
features = [
    feature for feature in features_raw
    if get_iso3(feature) and get_iso3(feature) != "-99" and get_iso3(feature) in country_lookup
]

print("Feature count:", feature_count(countries))
print("Property names:", property_names(countries))
print("Playable countries:", len(features))
print("Flag/name misses:", misses)
print("First playable country:", get_country_name(features[0]), get_iso3(features[0]))


Feature count: 258
Property names: ['ISO3166-1-Alpha-2', 'ISO3166-1-Alpha-3', 'name']
Playable countries: 236
Flag/name misses: []
First playable country: Indonesia IDN


In [12]:
# Build sorted country options for the searchable combobox.
country_options = sorted(
    (country_lookup[get_iso3(feature)]["name"], get_iso3(feature))
    for feature in features
)

name_to_iso3 = {name: iso3 for name, iso3 in country_options}
iso3_to_name = {iso3: name for name, iso3 in country_options}

country_names = [name for name, iso3 in country_options]
country_names[:10]


['Afghanistan',
 'Aland',
 'Albania',
 'Algeria',
 'American Samoa',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antarctica',
 'Antigua and Barbuda']

In [13]:
def one_feature_collection(feature):
    """Wrap one feature so ipyleaflet can draw it cleanly."""
    return {"type": "FeatureCollection", "features": [feature]}


def start_worldle(seed=8, max_guesses=6):
    """Create and display one playable Worldle game."""
    game = WorldleGame(features, seed=seed, max_guesses=max_guesses)
    target_data = one_feature_collection(game.target)

    mystery_style = {
        "color": "#1d3557",
        "fillColor": "#e63946",
        "weight": 2,
        "fillOpacity": 0.58,
    }

    m = make_map(center=(20, 0), zoom=2)
    add_geojson(m, target_data, name="Mystery country", style=mystery_style)
    fit_map_to_geojson(m, target_data)

    title = widgets.HTML(
        """
        <div style='font-family:-apple-system, BlinkMacSystemFont, Segoe UI, sans-serif;
                    padding:10px 0 4px 0'>
            <h2 style='margin:0;color:#1d3557'>🌎 Worldle Notebook</h2>
            <p style='margin:4px 0;color:#555'>Guess the mystery country. The arrow points from your guess toward the target.</p>
        </div>
        """
    )

    combobox = widgets.Combobox(
        placeholder="Type a country name",
        options=country_names,
        description="Guess:",
        ensure_option=True,
        layout=widgets.Layout(width="420px"),
    )

    guess_button = widgets.Button(
        description="Guess",
        button_style="primary",
        icon="check",
        layout=widgets.Layout(width="110px"),
    )

    give_up_button = widgets.Button(
        description="Give up",
        button_style="warning",
        icon="flag",
        layout=widgets.Layout(width="110px"),
    )

    banner = widgets.HTML(
        f"<div style='padding:8px;color:#555'>Guesses left: {max_guesses}</div>"
    )
    history = widgets.HTML("")
    share = widgets.Textarea(
        value="",
        description="Share:",
        layout=widgets.Layout(width="100%", height="90px"),
        disabled=False,
    )

    rows = []

    def end_controls():
        guess_button.disabled = True
        give_up_button.disabled = True
        combobox.disabled = True
        share.value = game.result_string()

    def submit_guess(_=None):
        if game.finished:
            return

        guessed_name = combobox.value.strip()
        if guessed_name not in name_to_iso3:
            banner.value = "<div style='padding:8px;color:#b00020'>Please choose a country from the list.</div>"
            return

        guess_iso3 = name_to_iso3[guessed_name]
        feedback = game.submit_guess(guess_iso3)
        meta = country_lookup.get(guess_iso3, {})

        rows.append(
            render_guess_row(
                feedback["guess_name"],
                meta.get("flag_path"),
                feedback["arrow"],
                feedback["distance_km"],
            )
        )
        history.value = "".join(rows)

        if feedback["correct"]:
            banner.value = (
                "<div style='padding:10px;border-radius:10px;background:#d8f3dc;color:#1b4332;'>"
                f"<h2 style='margin:0'>🎉 Correct! It was {feedback['target_name']}.</h2>"
                "</div>"
            )
            end_controls()
        elif game.finished:
            answer = get_country_name(game.target)
            banner.value = (
                "<div style='padding:10px;border-radius:10px;background:#ffe5d9;color:#7f0000;'>"
                f"<h3 style='margin:0'>Game over. The country was {answer}.</h3>"
                "</div>"
            )
            end_controls()
        else:
            remaining = max_guesses - len(game.guesses)
            banner.value = (
                f"<div style='padding:8px;color:#555'>"
                f"{format_feedback(feedback)}<br>Guesses left: {remaining}</div>"
            )

        combobox.value = ""

    def give_up(_=None):
        answer = game.give_up()
        banner.value = (
            "<div style='padding:10px;border-radius:10px;background:#fff3cd;color:#664d03;'>"
            f"<h3 style='margin:0'>The mystery country was {answer}.</h3>"
            "</div>"
        )
        end_controls()

    guess_button.on_click(submit_guess)
    give_up_button.on_click(give_up)
    combobox.on_submit(submit_guess)

    controls = widgets.HBox([combobox, guess_button, give_up_button])
    layout = widgets.VBox([title, m, controls, banner, history, share])
    display(layout)
    return game


In [15]:
# Run this cell to play. Change the seed to get a different mystery country.
game = start_worldle(seed=9, max_guesses=6)


/tmp/ipykernel_10906/53497415.py:131: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  combobox.on_submit(submit_guess)
